# MongoDB Atlas Connector Sample

Load a MongoDB Atlas collection into a Delta table: a full load on the first run, then incremental loads on a Date watermark field. Read-only against MongoDB. Replace all `<PLACEHOLDER>` values before running.

## Prerequisites

1. **Helper.** Upload `mongodb_client.py` (the only helper file, from this folder) to a workspace folder and set `HELPER_DIR` below to that folder.
2. **Connector jars.** Download these five jars from Maven Central (links and SHA-256 values in `README.md`): `mongo-spark-connector_2.12-10.7.0.jar`, `mongodb-driver-sync-5.1.4.jar`, `mongodb-driver-core-5.1.4.jar`, `bson-5.1.4.jar` and `bson-record-codec-5.1.4.jar`. Upload them to a workspace folder, install each from the cluster's **Library** tab (**Install Library**, one at a time, waiting for each to finish), then restart the cluster.
3. **Credential.** In the AIDP Credential Store, create a **Secret Token** credential with key `uri` whose value is the connection string, `mongodb+srv://<user>:<password>@<cluster>.mongodb.net/`, for a read-only database user. Set `CREDENTIAL_NAME` below to its name. Do not type the URI into a cell: it contains the password.
4. **Network.** Atlas only accepts connections from IPs on the project's IP access list. Find the cluster's outbound IP with `urllib.request.urlopen("https://checkip.amazonaws.com").read()` in a scratch cell (do not save the output), add it under **Network Access**, and check it again after a cluster restart.

## Configuration

In [ ]:
import sys

HELPER_DIR = "<WORKSPACE_PATH_TO_HELPER_FOLDER>"  # contains mongodb_client.py
CREDENTIAL_NAME = "<CREDENTIAL_NAME>"  # Credential Store entry (Secret Token, key "uri")
TARGET = "<CATALOG>.<SCHEMA>.<TABLE>"
DATABASE = "<DATABASE>"
COLLECTION = "<COLLECTION>"
WATERMARK_FIELD = "<DATE_FIELD>"  # must hold BSON Dates (not strings); None = full re-read every run
STRING_FIELDS = []                # fields whose type differs between documents, read as text
SAMPLE_SIZE = 10000               # first run only: documents sampled to infer the schema
OVERLAP_SECONDS = 300
sys.path.insert(0, HELPER_DIR)

## Check the Connection

Reads the URI from the Credential Store and, for a `mongodb+srv://` URI, looks up its SRV and TXT records on the driver, because executors cannot. It then reads one sampled document: this proves network access, the IP access list, auth and read access, and fails with a hint instead of a bare Py4J error.

In [ ]:
import mongodb_client as m

uri = m.validate_uri(aidputils.secrets.get(name=CREDENTIAL_NAME, key="uri"))
# Executors cannot resolve mongodb+srv:// DNS records, so resolve them here, on the driver.
uri = m.resolve_srv(spark, uri)
m.check_connection(spark, uri, DATABASE, COLLECTION)
print("connection ok")

## Read Documents

Later runs reuse the target table's schema, so a type the sample happens to guess differently cannot change the table under the `MERGE`. The watermark is read as epoch microseconds inside the helper: a collected TIMESTAMP comes back in the Python process's local timezone, which would shift it.

In [ ]:
from datetime import datetime, timezone

since, schema = None, None
if spark.catalog.tableExists(TARGET):
    schema = spark.table(TARGET).schema
    if WATERMARK_FIELD:
        since = m.latest_watermark(spark, TARGET, WATERMARK_FIELD)

until = datetime.now(timezone.utc) if WATERMARK_FIELD else None  # fixed for this run
df = m.read_collection(
    spark, uri, DATABASE, COLLECTION,
    schema=schema, string_fields=STRING_FIELDS, sample_size=SAMPLE_SIZE,
    watermark_field=WATERMARK_FIELD, since=since, until=until, overlap_seconds=OVERLAP_SECONDS,
)

## Write to Delta

The overlap window re-reads a few documents on purpose, so later runs `MERGE` on `_id` to remove duplicates. The actual MongoDB read happens here, so errors are explained here too.

In [ ]:
try:
    if not spark.catalog.tableExists(TARGET):
        df.write.format("delta").saveAsTable(TARGET)
    else:
        df.createOrReplaceTempView("incoming_document")
        spark.sql(
            f"MERGE INTO {TARGET} t USING incoming_document s ON t._id = s._id "
            "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *"
        )
except Exception as exc:
    raise m.explain_error(exc, uri) from None
print("documents in target:", spark.table(TARGET).count())

## Limits

- A watermark does not see deletes in MongoDB.
- `WATERMARK_FIELD` must hold BSON Dates. A string field never matches a Date bound, so an incremental read of it returns nothing.
- The schema is inferred once, on the first run, then widened (decimals to `decimal(38, >=10)`, ints to longs), because the connector's sampling otherwise silently nulls values wider than the sample. A field whose type varies between documents fails the read with `UnknownReason`: add it to `STRING_FIELDS`, drop the target table and rerun.
- Later runs reuse the target table's schema, so fields that first appear after the first run are not added. Drop the table and rerun to pick them up.
- To check a column for lost values, count on the target table: `df.filter(col.isNull())` is pushed down to MongoDB and counts only server-side nulls.
- A Spark task that fails with `UnknownReason` hides the real error. If even `df.limit(1).collect()` fails, the executors cannot use the connector: check that all five jars are installed as cluster libraries and the cluster was restarted.
- A free Atlas M0 cluster is for learning and testing, not production volume.